# Fine-tune BERT for Named Entity Recognition (NER)

This notebook loads a pre-trained BERT model, fine-tunes it on a token-classification NER task, evaluates with entity-level metrics, and runs inference on new text.

**Default dataset:** [CoNLL-2003](https://huggingface.co/datasets/conll2003) (English: PER, ORG, LOC, MISC).

**To use your own data:** prepare JSON/CSV with `tokens` (list of words) and `ner_tags` (integer label per token), or BIO strings — see the "Custom dataset" section at the end.

## 1. Install dependencies

In [ ]:
#!pip install -q datasets evaluate transformers accelerate seqeval

## 2. Load dataset and label names

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003", trust_remote_code=True)
dataset

In [ ]:
label_list = dataset["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
print("Labels:", label_list)

## 3. Load BERT tokenizer and model

We use `bert-base-uncased` with a token-classification head (one logit per label per token).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "bert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(device)

model.config.label2id, model.config.id2label

## 4. Tokenize and align labels

BERT uses WordPiece subwords. Labels must be aligned to the first subword of each word; other subwords get label `-100` (ignored in loss).

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    labels = []
    for i, label_ids in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids_aligned = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids_aligned.append(-100)
            elif word_idx != previous_word_idx:
                label_ids_aligned.append(label_ids[word_idx])
            else:
                label_ids_aligned.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids_aligned)
    tokenized["labels"] = labels
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## 5. Metrics (entity-level F1 with seqeval)

In [ ]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return seqeval.compute(predictions=true_predictions, references=true_labels)

## 6. Train

In [ ]:
from transformers import TrainingArguments, Trainer

output_dir = "./bert-ner-conll2003"

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="overall_f1",
    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
metrics

In [ ]:
save_path = "./bert-ner-checkpoint"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved to {save_path}")

## 7. Inference on new text

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model=save_path,
    tokenizer=save_path,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

text = "Apple Inc. was founded by Steve Jobs in Cupertino."
ner_pipeline(text)

## 8. (Optional) Custom dataset

Build a Hugging Face `Dataset` from your files. Each row needs:
- `tokens`: list of strings (one sentence)
- `ner_tags`: list of integers matching your `label2id`

Example with BIO strings:

In [ ]:
# from datasets import Dataset, DatasetDict
#
# custom_label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG"]
# label2id = {l: i for i, l in enumerate(custom_label_list)}
#
# def bio_to_ids(tags):
#     return [label2id[t] for t in tags]
#
# train_rows = [
#     {"tokens": ["John", "works", "at", "Google"], "ner_tags": bio_to_ids(["B-PER", "O", "O", "B-ORG"])},
# ]
# dataset = DatasetDict({
#     "train": Dataset.from_list(train_rows),
#     "validation": Dataset.from_list(train_rows),
# })
# id2label = {i: l for i, l in enumerate(custom_label_list)}
# Then re-run sections 3–7 with this `dataset` and `label_list = custom_label_list`.